# Análise exploratória — Assistência Técnica de Celulares

Análise da base de ordens de serviço (OS) já limpa (`dados/ordens_servico_limpo.csv`)
da **Opção Telefonia Celular** (Bauru/SP).

> Os dados são **simulados** com base em informações passadas pela empresa.

In [1]:
import pandas as pd
import plotly.express as px

# Lê a base limpa, já convertendo as colunas de data
df = pd.read_csv(
    "dados/ordens_servico_limpo.csv",
    sep=";",
    parse_dates=["data_entrada", "data_conclusao", "data_entrega"],
)

# OS entregues: usadas para receita, lucro e taxa de retorno
entregues = df[df["status"] == "Entregue"]

## 1. Visão geral da base

In [2]:
print("Total de OS:", len(df))
print("Clientes diferentes:", df["cliente_id"].nunique())
print("Marcas:", df["marca"].nunique(), "| Modelos diferentes:", df["modelo"].nunique())
print("Período:", df["data_entrada"].min().date(), "a", df["data_entrada"].max().date())
print("\nOS por status:")
print(df["status"].value_counts())
df.head()

Total de OS: 2519
Clientes diferentes: 1912
Marcas: 7 | Modelos diferentes: 173
Período: 2025-11-01 a 2026-10-17

OS por status:
status
Entregue              2082
Orçamento recusado     259
Cancelado              114
Pronto                  36
Em andamento            24
Aguardando peça          4
Name: count, dtype: int64


,os_id,data_entrada,data_conclusao,data_entrega,cliente_id,origem_cliente,marca,modelo,defeito,tipo_reparo,tecnico,status,valor,custo_peca,retorno_garantia,dias_reparo,mes,dia_semana,lucro
0,OS-0001,2025-11-01,2025-11-01,2025-11-02,CLI-0001,Indicação,Samsung,Galaxy S21,Software travando,Reinstalação de software,Felipe,Entregue,89.13,0.00,Sim,0.0,2025-11,Sábado,89.13
1,OS-0002,2025-11-01,2025-11-02,2025-11-07,CLI-0002,Indicação,Xiaomi,Redmi Note 11,Tela quebrada,Troca de tela,Carlos Eduardo,Entregue,534.28,295.42,Não,1.0,2025-11,Sábado,238.86
2,OS-0003,2025-11-01,2025-11-03,2025-11-03,CLI-0003,Passou na frente,Apple,iPhone 13,Tela quebrada,Troca de tela,Carlos Eduardo,Entregue,552.88,246.69,Não,2.0,2025-11,Sábado,306.19
3,OS-0004,2025-11-01,2025-11-07,2025-11-11,CLI-0001,Indicação,Xiaomi,Redmi Note 12,Não liga,Reparo de placa,Felipe,Entregue,215.79,47.59,Não,6.0,2025-11,Sábado,168.20
4,OS-0005,2025-11-01,2025-11-02,2025-11-03,CLI-0004,Indicação,Xiaomi,Redmi 12,Tela quebrada,Troca de tela,Felipe,Entregue,482.67,197.48,Não,1.0,2025-11,Sábado,285.19


In [3]:
fig = px.pie(df, names="status", title="OS por status", hole=0.4)
fig.show()

A base tem cerca de 2,5 mil OS em quase 12 meses. A maioria já foi **entregue**; cerca de 10%
tiveram o **orçamento recusado** pelo cliente e 4,5% foram canceladas depois de aprovadas.

## 2. Aparelhos recebidos por mês e por dia da semana

In [4]:
por_mes = df.groupby("mes").size().reset_index(name="aparelhos")

fig = px.bar(por_mes, x="mes", y="aparelhos", title="Aparelhos recebidos por mês", text="aparelhos")
fig.update_layout(xaxis_title="Mês", yaxis_title="Aparelhos")
fig.show()

In [5]:
# Compara dezembro/janeiro com a média dos outros meses completos (fev a set)
pico = por_mes[por_mes["mes"].isin(["2025-12", "2026-01"])]["aparelhos"].mean()
normal = por_mes[(por_mes["mes"] >= "2026-02") & (por_mes["mes"] <= "2026-09")]["aparelhos"].mean()
print(f"Média dez/jan: {pico:.0f} | Média fev-set: {normal:.0f} | Diferença: {pico / normal - 1:.0%}")

Média dez/jan: 282 | Média fev-set: 205 | Diferença: 38%


In [6]:
ordem = ["Segunda", "Terça", "Quarta", "Quinta", "Sexta", "Sábado"]
por_dia = df["dia_semana"].value_counts().reindex(ordem).reset_index()
por_dia.columns = ["dia", "aparelhos"]

fig = px.bar(por_dia, x="dia", y="aparelhos", text="aparelhos", title="Aparelhos recebidos por dia da semana")
fig.update_layout(xaxis_title="", yaxis_title="Aparelhos")
fig.show()

Existe **pico em dezembro e janeiro** (fim de ano e férias), com bem mais aparelhos que a média
dos outros meses. Outubro aparece baixo só porque a base vai até 17/10.
Na semana, **segunda e sábado** são os dias mais cheios: segunda recebe o que quebrou no fim de
semana, e sábado é quando o cliente tem tempo de ir à loja.

## 3. Defeitos e modelos mais frequentes

In [7]:
defeitos = df["defeito"].value_counts().reset_index()
defeitos.columns = ["defeito", "quantidade"]
defeitos["percentual"] = (defeitos["quantidade"] / len(df) * 100).round(1)

fig = px.bar(defeitos, x="quantidade", y="defeito", orientation="h",
             title="Defeitos mais frequentes", text="percentual")
fig.update_traces(texttemplate="%{text}%")
fig.update_layout(yaxis={"categoryorder": "total ascending"}, xaxis_title="OS", yaxis_title="")
fig.show()
defeitos

,defeito,quantidade,percentual
0,Tela quebrada,905,35.9
1,Bateria viciada,535,21.2
2,Conector de carga,370,14.7
3,Danos por água,245,9.7
4,Não liga,239,9.5
5,Software travando,225,8.9


In [8]:
# 10 modelos que mais chegam com tela quebrada (ajuda a decidir o estoque de telas)
telas = df[df["defeito"] == "Tela quebrada"]["modelo"].value_counts().head(10).reset_index()
telas.columns = ["modelo", "aparelhos"]
fig = px.bar(telas, x="aparelhos", y="modelo", orientation="h", text="aparelhos",
             title="Modelos que mais chegam com tela quebrada")
fig.update_layout(yaxis={"categoryorder": "total ascending"}, xaxis_title="Aparelhos", yaxis_title="")
fig.show()

**Tela quebrada** é de longe o defeito mais comum (mais de um terço das OS), seguida de
bateria viciada. A base tem mais de 170 modelos diferentes, mas poucos concentram a maior parte
das telas quebradas: são esses que vale a pena ter em estoque.

## 4. Tempo médio de reparo por tipo de reparo

In [9]:
tempo = df.groupby("tipo_reparo")["dias_reparo"].mean().round(1).reset_index()
tempo = tempo.sort_values("dias_reparo")

fig = px.bar(tempo, x="dias_reparo", y="tipo_reparo", orientation="h",
             title="Tempo médio de reparo (dias)", text="dias_reparo")
fig.update_layout(xaxis_title="Dias", yaxis_title="")
fig.show()
print(f"Tempo médio geral: {df['dias_reparo'].mean():.1f} dias")

Tempo médio geral: 2.5 dias


**Reparo de placa** e **limpeza de oxidação** são os mais demorados (5 a 6 dias em média).
Software, bateria e conector ficam prontos em 1 a 2 dias. A troca de tela às vezes atrasa
porque fica esperando peça.

## 5. Taxa de retorno na garantia

In [10]:
# Taxa de retorno = OS que voltaram na garantia / OS entregues
def taxa_retorno(coluna):
    taxa = entregues.groupby(coluna)["retorno_garantia"].apply(lambda s: (s == "Sim").mean() * 100)
    return taxa.round(1).sort_values().reset_index(name="taxa_retorno")

print(f"Taxa de retorno geral: {(entregues['retorno_garantia'] == 'Sim').mean():.1%}")

retorno_defeito = taxa_retorno("defeito")
fig = px.bar(retorno_defeito, x="taxa_retorno", y="defeito", orientation="h",
             title="Taxa de retorno por defeito (%)", text="taxa_retorno")
fig.update_layout(xaxis_title="% de retorno", yaxis_title="")
fig.show()

Taxa de retorno geral: 8.1%


In [11]:
retorno_tecnico = taxa_retorno("tecnico")
fig = px.bar(retorno_tecnico, x="taxa_retorno", y="tecnico", orientation="h",
             title="Taxa de retorno por técnico (%)", text="taxa_retorno")
fig.update_layout(xaxis_title="% de retorno", yaxis_title="")
fig.show()

**Danos por água** é o defeito que mais volta (mais de um quarto dos aparelhos), seguido de
"Não liga" (reparo de placa). Os dois técnicos, Felipe e Carlos Eduardo, têm taxas de retorno
parecidas: o retorno depende mais do defeito do que de quem conserta.

## 6. Receita, lucro e margem por tipo de reparo

In [12]:
por_tipo = entregues.groupby("tipo_reparo")[["valor", "lucro"]].sum()
por_tipo["margem_%"] = (por_tipo["lucro"] / por_tipo["valor"] * 100).round(1)
por_tipo["parte_receita_%"] = (por_tipo["valor"] / por_tipo["valor"].sum() * 100).round(1)
por_tipo = por_tipo.sort_values("valor").reset_index()

fig = px.bar(por_tipo, x=["valor", "lucro"], y="tipo_reparo", orientation="h", barmode="group",
             title="Receita e lucro por tipo de reparo (R$)")
fig.update_layout(xaxis_title="R$", yaxis_title="", legend_title="")
fig.show()

print(f"Receita total: R$ {entregues['valor'].sum():,.2f}")
print(f"Lucro bruto:   R$ {entregues['lucro'].sum():,.2f} (receita menos custo das peças)")
print(f"Margem:        {entregues['lucro'].sum() / entregues['valor'].sum():.1%}")
print(f"Ticket médio:  R$ {entregues['valor'].mean():,.2f}")
por_tipo

Receita total: R$ 751,943.98
Lucro bruto:   R$ 460,169.30 (receita menos custo das peças)
Margem:        61.2%
Ticket médio:  R$ 361.16


,tipo_reparo,valor,lucro,margem_%,parte_receita_%
0,Reinstalação de software,23371.58,23371.58,100.0,3.1
1,Limpeza de oxidação,51257.34,45866.73,89.5,6.8
2,Troca de conector,54895.36,42843.43,78.0,7.3
3,Reparo de placa,74074.36,58816.85,79.4,9.9
4,Troca de bateria,112285.32,70685.41,63.0,14.9
5,Troca de tela,436060.02,218585.30,50.1,58.0


**Troca de tela** traz mais da metade da receita, mas tem a **menor margem** (cerca de 50%),
porque a tela é a peça mais cara. Serviços que quase não usam peça (software, oxidação, placa)
têm margem alta, mas faturam pouco.

## 7. Orçamentos recusados

In [13]:
recusa = df.groupby("tipo_reparo")["status"].apply(lambda s: (s == "Orçamento recusado").mean() * 100)
recusa = recusa.round(1).sort_values().reset_index(name="recusados_%")

fig = px.bar(recusa, x="recusados_%", y="tipo_reparo", orientation="h", text="recusados_%",
             title="Orçamentos recusados por tipo de reparo (%)")
fig.update_layout(xaxis_title="% recusado", yaxis_title="")
fig.show()

recusados = df[df["status"] == "Orçamento recusado"]
print(f"Orçamentos recusados: {len(recusados)} ({len(recusados) / len(df):.1%} das OS)")
print(f"Valor que não fechou: R$ {recusados['valor'].sum():,.2f}")

Orçamentos recusados: 259 (10.3% das OS)
Valor que não fechou: R$ 99,589.03


**Reparo de placa** e **limpeza de oxidação** têm cerca de 27% dos orçamentos recusados, bem acima
dos outros serviços (4% a 8%). São reparos caros e sem garantia de que o aparelho volta a funcionar.

## 8. Clientes

In [14]:
df = df.sort_values(["data_entrada", "os_id"])
df["cliente_voltou"] = df.groupby("cliente_id").cumcount() > 0  # já tinha vindo antes

os_por_cliente = df.groupby("cliente_id").size()
print(f"Clientes diferentes: {len(os_por_cliente)}")
print(f"Clientes que voltaram (2+ OS): {(os_por_cliente > 1).sum()} ({(os_por_cliente > 1).mean():.1%})")

novos = df[~df["cliente_voltou"]]
origem = novos["origem_cliente"].value_counts().reset_index()
origem.columns = ["origem", "clientes"]
fig = px.pie(origem, names="origem", values="clientes", hole=0.4, title="Como os clientes novos conheceram a loja")
fig.show()

Clientes diferentes: 1912
Clientes que voltaram (2+ OS): 459 (24.0%)


Cerca de **um quarto dos clientes volta** com outro aparelho. A **indicação** é a principal forma
de chegada de clientes novos (35%), seguida do Instagram.

## 9. Conclusões

As conclusões abaixo são **calculadas a partir dos dados**: se a base mudar, os números mudam junto.

In [15]:
from IPython.display import Markdown

def pct(x):
    return f"{x * 100:.1f}".replace(".", ",") + "%"

def reais(x):
    return "R$ " + f"{x:,.0f}".replace(",", ".")

# 1. Carro-chefe: maior receita e a margem dele
por_tipo = entregues.groupby("tipo_reparo")[["valor", "lucro"]].sum()
top = por_tipo["valor"].idxmax()
parte = por_tipo.loc[top, "valor"] / por_tipo["valor"].sum()
margem_top = por_tipo.loc[top, "lucro"] / por_tipo.loc[top, "valor"]
margem_geral = entregues["lucro"].sum() / entregues["valor"].sum()
modelos_tela = ", ".join(df[df["defeito"] == "Tela quebrada"]["modelo"].value_counts().head(5).index)

# 2. Pico de dez/jan
variacao_pico = pico / normal - 1

# 3. Defeito que mais volta
ret = entregues.groupby("defeito")["retorno_garantia"].apply(lambda s: (s == "Sim").mean())
ret_geral = (entregues["retorno_garantia"] == "Sim").mean()
vezes = f"{ret.max() / ret_geral:.1f}".replace(".", ",")

# 4. Orçamentos recusados
taxa_recusa = df.groupby("tipo_reparo")["status"].apply(lambda s: (s == "Orçamento recusado").mean())
valor_recusado = df[df["status"] == "Orçamento recusado"]["valor"].sum()

# 5. Origem dos clientes novos e recorrência
origem_top = novos["origem_cliente"].value_counts(normalize=True)
voltaram = (os_por_cliente > 1).mean()

Markdown(f"""
1. **{top} é o carro-chefe, mas tem a menor margem:** {pct(parte)} da receita, com margem de
   {pct(margem_top)} (média geral: {pct(margem_geral)}). Vale manter estoque das telas mais procuradas
   ({modelos_tela}) e negociar o preço com fornecedores.
2. **Dezembro e janeiro são o pico:** em média {pico:.0f} aparelhos/mês, contra {normal:.0f} nos meses
   normais ({pct(variacao_pico)} a mais). Bom momento para reforçar a equipe.
3. **{ret.idxmax()} volta muito:** {pct(ret.max())} dos aparelhos entregues voltam na garantia, contra
   {pct(ret_geral)} na média geral ({vezes}× mais). Vale rever o procedimento ou
   avisar o cliente do risco.
4. **{taxa_recusa.idxmax()} perde orçamentos:** {pct(taxa_recusa.max())} dos orçamentos recusados;
   no total, {reais(valor_recusado)} em orçamentos não fecharam. Vale rever o preço ou oferecer parcelamento.
5. **{origem_top.index[0]} é o melhor canal:** traz {pct(origem_top.iloc[0])} dos clientes novos, e
   {pct(voltaram)} dos clientes voltam. Um programa de indicação pode trazer ainda mais clientes.
""")


1. **Troca de tela é o carro-chefe, mas tem a menor margem:** 58,0% da receita, com margem de
   50,1% (média geral: 61,2%). Vale manter estoque das telas mais procuradas
   (iPhone 11, Galaxy A14, Moto G54, iPhone 13, Galaxy A54) e negociar o preço com fornecedores.
2. **Dezembro e janeiro são o pico:** em média 282 aparelhos/mês, contra 205 nos meses
   normais (38,1% a mais). Bom momento para reforçar a equipe.
3. **Danos por água volta muito:** 27,3% dos aparelhos entregues voltam na garantia, contra
   8,1% na média geral (3,4× mais). Vale rever o procedimento ou
   avisar o cliente do risco.
4. **Limpeza de oxidação perde orçamentos:** 27,8% dos orçamentos recusados;
   no total, R$ 99.589 em orçamentos não fecharam. Vale rever o preço ou oferecer parcelamento.
5. **Indicação é o melhor canal:** traz 35,3% dos clientes novos, e
   24,0% dos clientes voltam. Um programa de indicação pode trazer ainda mais clientes.
